In [ ]:
import collections
import csv
import os
import tkinter as tk
from tkinter import ttk

# =====================================================================
# CONFIGURATION
# =====================================================================
INPUT_FILE = "maze.csv"                       # Path to input maze CSV
OUTPUT_FILE = "shortest_path_output.csv"      # Output path with 'x' marks
T_KEYS = 2                                    # Number of key/door pairs (keys 'a', 'b')
TARGET_CHAR = "C"                             # Destination character to reach

# Visual styling
KEY_DOOR_COLORS = {
    0: "#E6194B",  # a / A (Red)
    1: "#3CB44B",  # b / B (Green)
    2: "#4363D8",  # c / C (Blue)
    3: "#F58231",  # d / D (Orange)
    4: "#911EB4",  # e / E (Purple)
    5: "#46F0F0",  # f / F (Cyan)
}

CELL_COLORS = {
    ".": "#FFFFFF",
    "#": "#1E1E1E",
    "S": "#2ECC71",
}


# =====================================================================
# 1. FILE LOAD & EXPORT
# =====================================================================
def load_maze_from_csv(filename):
    """Loads and sanitizes the maze matrix from a CSV file."""
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Cannot find input file: {os.path.abspath(filename)}")

    grid = []
    with open(filename, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if not row:
                continue
            cleaned_row = [cell.strip().strip('"').strip("'") for cell in row]
            if cleaned_row:
                grid.append(cleaned_row)
    return grid


def export_solution_csv(grid, path_states, filename="shortest_path_output.csv"):
    """Exports the solved maze, marking steps with 'x'."""
    out_grid = [row[:] for row in grid]
    path_coords = set((r, c) for r, c, _ in path_states)

    for r, c in path_coords:
        cell = grid[r][c]
        if cell in ("S", "E", TARGET_CHAR):
            continue
        elif "a" <= cell <= "f" or "A" <= cell <= "F":
            out_grid[r][c] = f"x({cell})"
        else:
            out_grid[r][c] = "x"

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(out_grid)
    print(f"[✓] Output saved to: {os.path.abspath(filename)}")


# =====================================================================
# 2. BITMASK BFS SOLVER
# =====================================================================
def solve_maze(grid, t, target_char="C"):
    """
    Finds optimal shortest path from 'S' to target_char using Bitmask BFS.
    Time Complexity: O(N^2 * 2^T)
    Space Complexity: O(N^2 * 2^T)
    """
    n_rows = len(grid)
    n_cols = len(grid[0])
    start_pos = None
    target_pos = None

    for r in range(n_rows):
        for c in range(n_cols):
            if grid[r][c] == "S":
                start_pos = (r, c)
            elif grid[r][c] == target_char:
                target_pos = (r, c)

    if not start_pos:
        print("[!] Error: 'S' (Start) not found in grid!")
        return -1, []
    if not target_pos:
        print(f"[!] Warning: Target '{target_char}' not found in grid!")
        return -1, []

    start_mask = 0
    q = collections.deque([(start_pos[0], start_pos[1], start_mask)])
    visited = set([(start_pos[0], start_pos[1], start_mask)])
    parent = {}
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    target_state = None

    while q:
        r, c, mask = q.popleft()

        if (r, c) == target_pos:
            target_state = (r, c, mask)
            break

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < n_rows and 0 <= nc < n_cols):
                continue

            cell = grid[nr][nc]
            if cell == "#":
                continue

            nmask = mask

            # Check locked door (only blocks if it is NOT the explicit target)
            if "A" <= cell <= "F" and cell != target_char:
                door_idx = ord(cell) - ord("A")
                if door_idx < t:
                    if not ((mask >> door_idx) & 1):
                        continue

            # Pick up key
            if "a" <= cell <= "f":
                key_idx = ord(cell) - ord("a")
                if key_idx < t:
                    nmask |= (1 << key_idx)

            next_state = (nr, nc, nmask)
            if next_state not in visited:
                visited.add(next_state)
                parent[next_state] = (r, c, mask)
                q.append(next_state)

    if not target_state:
        return -1, []

    # Reconstruct optimal path
    path = []
    curr = target_state
    while curr in parent:
        path.append(curr)
        curr = parent[curr]
    path.append((start_pos[0], start_pos[1], start_mask))
    path.reverse()

    return len(path) - 1, path


# =====================================================================
# 3. INTERACTIVE TKINTER DESKTOP APPLICATION
# =====================================================================
class InteractiveMazeApp:
    def __init__(self, root, file_path, t, initial_target="C"):
        self.root = root
        self.root.title("Dynamic Bitmask BFS Maze Visualizer")
        self.file_path = file_path
        self.t = t
        self.current_target = initial_target

        self.path = []
        self.step_idx = 0
        self.anim_job = None

        # Build UI layout
        self._build_controls()
        self._build_inventory_hud()

        # Canvas placeholder
        self.canvas = None
        self.cell_rects = {}
        self.text_items = {}

        # Force window to foreground
        self.root.lift()
        self.root.attributes("-topmost", True)
        self.root.after_idle(self.root.attributes, "-topmost", False)
        self.root.focus_force()

        # Initial load and solve
        self.reload_from_disk_and_solve()

    def _build_controls(self):
        ctrl_frame = ttk.Frame(self.root, padding=8)
        ctrl_frame.pack(fill=tk.X)

        ttk.Label(ctrl_frame, text="Target:", font=("Helvetica", 10, "bold")).pack(side=tk.LEFT, padx=3)
        self.target_entry = ttk.Entry(ctrl_frame, width=4, font=("Consolas", 11, "bold"))
        self.target_entry.insert(0, self.current_target)
        self.target_entry.pack(side=tk.LEFT, padx=3)

        self.reload_btn = ttk.Button(ctrl_frame, text="🔄 Reload File & Run", command=self.reload_from_disk_and_solve)
        self.reload_btn.pack(side=tk.LEFT, padx=5)

        self.status_lbl = ttk.Label(ctrl_frame, text="Ready", font=("Helvetica", 10))
        self.status_lbl.pack(side=tk.RIGHT, padx=10)

    def _build_inventory_hud(self):
        self.hud_frame = ttk.Frame(self.root, padding=6)
        self.hud_frame.pack(fill=tk.X)

        ttk.Label(self.hud_frame, text="Keys Collected: ", font=("Helvetica", 10, "bold")).pack(side=tk.LEFT)
        self.inventory_labels = {}
        for i in range(self.t):
            k = chr(ord("a") + i)
            lbl = tk.Label(
                self.hud_frame,
                text=f" {k.upper()} ",
                fg="#888888",
                bg="#DDDDDD",
                relief=tk.SUNKEN,
                font=("Consolas", 10, "bold"),
            )
            lbl.pack(side=tk.LEFT, padx=3)
            self.inventory_labels[i] = lbl

    def reload_from_disk_and_solve(self):
        if self.anim_job:
            self.root.after_cancel(self.anim_job)
            self.anim_job = None

        self.current_target = self.target_entry.get().strip()

        # Read directly from disk so edits are instantly reflected
        try:
            self.grid = load_maze_from_csv(self.file_path)
        except Exception as e:
            self.status_lbl.config(text=f"Load Error: {e}")
            return

        self.n_rows = len(self.grid)
        self.n_cols = len(self.grid[0])
        self.cell_size = max(24, min(48, 620 // max(self.n_rows, self.n_cols)))

        # Reset inventory buttons
        for i in range(self.t):
            self.inventory_labels[i].config(bg="#DDDDDD", fg="#888888", relief=tk.SUNKEN)

        # Solve
        path_len, self.path = solve_maze(self.grid, t=self.t, target_char=self.current_target)

        # Render grid
        self.render_grid()

        if path_len == -1:
            self.status_lbl.config(text=f"No path found to '{self.current_target}'!")
            return

        self.status_lbl.config(text=f"Path found! Length: {path_len}")
        export_solution_csv(self.grid, self.path, OUTPUT_FILE)

        self.step_idx = 0
        self.animate_step()

    def render_grid(self):
        c_width = self.n_cols * self.cell_size
        c_height = self.n_rows * self.cell_size

        if self.canvas is None:
            self.canvas = tk.Canvas(self.root, width=c_width, height=c_height, bg="#FFFFFF")
            self.canvas.pack(pady=5)
        else:
            self.canvas.config(width=c_width, height=c_height)
            self.canvas.delete("all")

        self.cell_rects.clear()
        self.text_items.clear()

        for r in range(self.n_rows):
            for c in range(self.n_cols):
                x1, y1 = c * self.cell_size, r * self.cell_size
                x2, y2 = x1 + self.cell_size, y1 + self.cell_size
                cell = self.grid[r][c]

                fill = CELL_COLORS.get(cell, "#FFFFFF")
                if cell == self.current_target:
                    fill = "#E74C3C"  # Target highlighted in Red
                elif "a" <= cell <= "f":
                    fill = KEY_DOOR_COLORS.get(ord(cell) - ord("a"), "#DDDDDD")
                elif "A" <= cell <= "F":
                    fill = KEY_DOOR_COLORS.get(ord(cell) - ord("A"), "#DDDDDD")

                rect = self.canvas.create_rectangle(x1, y1, x2, y2, fill=fill, outline="#C0C0C0", width=1)
                self.cell_rects[(r, c)] = rect

                text = cell if cell not in (".", "#") else ""
                txt_id = self.canvas.create_text(
                    (x1 + x2) // 2,
                    (y1 + y2) // 2,
                    text=text,
                    font=("Helvetica", max(9, self.cell_size // 3), "bold"),
                    fill="#FFFFFF" if cell in ("#", "E", self.current_target) or cell.isupper() else "#000000",
                )
                self.text_items[(r, c)] = txt_id

    def animate_step(self):
        if self.step_idx >= len(self.path):
            self.status_lbl.config(text=f"Arrived at {self.current_target}! Total steps: {len(self.path) - 1}")
            return

        r, c, mask = self.path[self.step_idx]
        self.status_lbl.config(text=f"Step: {self.step_idx}/{len(self.path) - 1} | Pos: ({r}, {c})")

        # Inventory HUD update
        for i in range(self.t):
            if (mask >> i) & 1:
                self.inventory_labels[i].config(bg=KEY_DOOR_COLORS[i], fg="#FFFFFF", relief=tk.RAISED)

        cell = self.grid[r][c]
        if "a" <= cell <= "f":
            self.canvas.itemconfig(self.cell_rects[(r, c)], fill="#FFFFFF")
            self.canvas.itemconfig(self.text_items[(r, c)], text=f"✓{cell}", fill="#888888")
        elif "A" <= cell <= "F" and cell != self.current_target:
            self.canvas.itemconfig(self.cell_rects[(r, c)], fill="#FFFFFF")
            self.canvas.itemconfig(self.text_items[(r, c)], text=f"🔓{cell}", fill="#27AE60")

        if cell not in ("S", self.current_target):
            self.canvas.itemconfig(self.cell_rects[(r, c)], outline="#F1C40F", width=2)

        # Agent position dot
        self.canvas.delete("agent")
        self.canvas.create_oval(
            c * self.cell_size + self.cell_size * 0.25,
            r * self.cell_size + self.cell_size * 0.25,
            c * self.cell_size + self.cell_size * 0.75,
            r * self.cell_size + self.cell_size * 0.75,
            fill="#3498DB",
            outline="",
            tags="agent",
        )

        self.step_idx += 1
        self.anim_job = self.root.after(160, self.animate_step)


# =====================================================================
# 4. EXECUTION
# =====================================================================
# Close previous tk instances if re-running cell
try:
    root.destroy()
except Exception:
    pass

root = tk.Tk()
app = InteractiveMazeApp(root, file_path=INPUT_FILE, t=T_KEYS, initial_target=TARGET_CHAR)
root.mainloop()

[✓] Output saved to: c:\Users\Mohammed Muheet Pash\Desktop\DSA_HACK_P1\short_path_output.csv
